In [1]:
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 3.0.3
numpy: 2.4.6


In [2]:
bureau = pd.read_csv('../data/raw/bureau.csv')
bureau_balance = pd.read_csv('../data/raw/bureau_balance.csv')
previous_application = pd.read_csv('../data/raw/previous_application.csv')
pos_cash = pd.read_csv('../data/raw/POS_CASH_balance.csv')
credit_card = pd.read_csv('../data/raw/credit_card_balance.csv')
installments = pd.read_csv('../data/raw/installments_payments.csv')

for name, table in [('bureau', bureau), ('bureau_balance', bureau_balance),
                     ('previous_application', previous_application),
                     ('POS_CASH_balance', pos_cash),
                     ('credit_card_balance', credit_card),
                     ('installments_payments', installments)]:
    print(f'=== {name} ===')
    print('Shape:', table.shape)
    print('Columns:', table.columns.tolist())
    print()

=== bureau ===
Shape: (1716428, 17)
Columns: ['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY']

=== bureau_balance ===
Shape: (27299925, 3)
Columns: ['SK_ID_BUREAU', 'MONTHS_BALANCE', 'STATUS']

=== previous_application ===
Shape: (1670214, 37)
Columns: ['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY', 'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'NAME_CASH_LOAN_PURPOSE', 'NAME_CONTRACT_STATUS', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE', 'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'N

In [3]:
print("bureau, linhas por cliente (média):", bureau.groupby('SK_ID_CURR').size().mean().round(2))
print("previous_application, linhas por cliente (média):", previous_application.groupby('SK_ID_CURR').size().mean().round(2))
print("POS_CASH_balance, linhas por SK_ID_PREV (média):", pos_cash.groupby('SK_ID_PREV').size().mean().round(2))

bureau, linhas por cliente (média): 5.61
previous_application, linhas por cliente (média): 4.93
POS_CASH_balance, linhas por SK_ID_PREV (média): 10.68


In [4]:
bureau['CREDIT_ACTIVE'].unique()

<StringArray>
['Closed', 'Active', 'Sold', 'Bad debt']
Length: 4, dtype: str

In [ ]:
# Aggregate bureau.csv to SK_ID_CURR level (one row per client).
# Combines count-based features (how many credits, how many active) with
# amount-based features (sum and mean of credit exposure and overdue amounts).

bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_CREDIT_COUNT=('SK_ID_BUREAU', 'count'),
    BUREAU_CREDIT_ACTIVE_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    BUREAU_DAYS_CREDIT_MEAN=('DAYS_CREDIT', 'mean'),
    BUREAU_AMT_CREDIT_SUM_MEAN=('AMT_CREDIT_SUM', 'mean'),
    BUREAU_AMT_CREDIT_SUM_TOTAL=('AMT_CREDIT_SUM', 'sum'),
    BUREAU_AMT_CREDIT_SUM_DEBT_MEAN=('AMT_CREDIT_SUM_DEBT', 'mean'),
    BUREAU_AMT_CREDIT_SUM_DEBT_TOTAL=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN=('AMT_CREDIT_SUM_OVERDUE', 'mean'),
    BUREAU_CREDIT_DAY_OVERDUE_MAX=('CREDIT_DAY_OVERDUE', 'max'),
    BUREAU_CNT_CREDIT_PROLONG_SUM=('CNT_CREDIT_PROLONG', 'sum'),
).reset_index()

print(bureau_agg.shape)
print(bureau_agg.head())